# Exercise

### 발견된 문제점 (우선순위 순)

[P1] refine 로직 버그 : 
refine 결과가 재평가되지 않고 바로 fallback으로 넘어가서, 사실상 refine이 무의미하게 동작함.

[P2] 검색 데이터 문제 : 
DB가 랜덤/관련 기반이 아니라 앞에서 잘라 쓰는 구조라, 필요한 정보가 없어서 RAG가 제대로 작동하지 않음.

[P3] context 불일치 : 
답변 생성은 6개 문서, 평가는 4개 문서로 해서 맞는 답도 틀렸다고 판단되는 문제 발생.

[P4] DB 구조 문제 : 
Chroma 기반 persist 구조처럼 설명했지만 실제는 FAISS in-memory라 매번 재임베딩 → 비효율적.

[P5] 출처 미사용 : 
검색에서 만든 citations를 최종 답변에 안 써서, 출처가 모델 추측으로 대체됨.

[P6] 검색 품질 필터 없음 : 
관련 없는 문서도 그대로 들어가서 잘못된 답(환각) 유발 가능.

[P7] 평가 방식 취약 : 
"YES" 문자열 포함 여부로 판단해서 한국어/다양한 응답에 취약하고 신뢰도 낮음.


### 개선 설계

**1. refine 루프 정상화 (P1)** — refine 후 반드시 grade로 재평가하고, 종료 판단은 grade에서:
```python
def decide_after_refine(state):      # refine → 항상 grade로 재평가
    return "grade"

def decide_after_grade(state):       # 종료/보정/fallback을 여기서 결정
    if state.get("grounded"):
        return END
    if int(state.get("refine_tries", 0)) >= MAX_REFINE_TRIES:
        return "fallback" if ENABLE_FALLBACK else END
    return "refine"
# grade의 conditional_edges에 "fallback" 매핑 추가 필요
```

**2. 코퍼스/검색 개선 (P2, P6)**
- `filtered_take`를 주제 관련 필터 또는 랜덤 셔플 샘플링으로 바꾸거나 `MAX_ARTICLES`를 크게 늘림.
- `as_retriever` 대신 `similarity_search_with_relevance_scores`로 임계값(예: 0.5) 미만이면 context 제거 → 근거 없으면 곧장 fallback.

**3. grade/generate context 일치 (P3)** — 두 노드가 동일한 상위 K개(예: `context[:TOP_K]`)를 사용하도록 통일. 채점은 생성에 실제 쓰인 context로.

**4. 영속화 (P4)** — 한쪽으로 통일. FAISS 유지 시:
```python
INDEX_DIR = "/content/faiss_index"
if os.path.exists(INDEX_DIR):
    vectordb = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
else:
    vectordb = build_faiss(all_docs); vectordb.save_local(INDEX_DIR)
```
그리고 미사용 Chroma 설치/`mkdir chroma`/`CHROMA_DIR` 제거(또는 실제 Chroma persist로 교체).

**5. 검증된 출처 주입 (P5)** — generate/최종 출력에서 모델 생성 출처 대신 `state["citations"]`를 중복 제거 후 append.

**6. grade 견고화 (P7)** — YES/NO를 강제하는 프롬프트 + 정규식 파싱, 가능하면 logprob/제약 디코딩. 또는 임베딩 유사도 기반 근거 판정으로 대체해 LLM 호출 절감.

<요약>

[P1] refine 루프 정상화 : 
refine 이후 바로 종료하지 말고 반드시 grade로 재평가하도록 변경. 종료/재시도/fallback 판단은 grade 단계에서만 수행하도록 구조 분리.

[P2] 코퍼스 및 검색 개선 : 
데이터를 앞에서 자르는 방식 대신 랜덤 샘플링 또는 주제 기반 필터링 적용. 유사도 점수 기준을 도입해 관련성 낮은 문서는 context에서 제거하고, 근거 없으면 즉시 fallback.

[P3] context 일관성 확보 : 
generate와 grade가 동일한 개수의 context를 사용하도록 통일하여, 실제 사용된 근거 기준으로 평가되게 수정.

[P4] 벡터 DB 영속화 : 
FAISS를 유지한다면 save/load를 적용해 재임베딩 방지. 또는 Chroma로 완전히 전환하되 현재 혼용된 설정은 제거.

[P5] 출처 정보 활용 : 
retrieve 단계에서 만든 citations를 최종 답변에 포함시켜, 모델이 임의로 생성한 출처 대신 검증된 메타데이터 사용.

[P6] 검색 품질 필터 추가 : 
유사도 threshold를 설정해 관련 없는 문서 유입 차단. 근거 부족 상황을 조기에 감지하도록 개선.

[P7] grade 판정 개선 : 
YES/NO 강제 출력 + 정규식 파싱으로 안정성 확보. 가능하면 LLM 평가 대신 임베딩 기반 유사도 판단으로 대체.


### 강사님 제공 기존 RAG

In [ ]:
!pip install -q chromadb langgraph langchain-community faiss-cpu

데이터를 저장할 폴더입니다.
- data : 원본 데이터를 저장합니다.
- chroma : 벡터 DB의 persist 디렉토리며 인덱스(임베딩 벡터)가 저장됩니다.

In [ ]:
!mkdir data chroma

mkdir: cannot create directory ‘data’: File exists
mkdir: cannot create directory ‘chroma’: File exists


기본 경로와 설정을 정의하겠습니다.  
변경하여 진행하여도 되지만 특히 모델 변경시 모델에 따른 인풋 데이터 포멧에 주의하여야합니다.

In [ ]:
import os, pathlib, re
from typing import Iterable, Optional, Dict

# 경로
DATA_DIR   = "/content/data"
CHROMA_DIR = "/content/chroma"

# 위키 데이터셋 설정
LANG = "ko" # 'en','ja' 등
DATASET_NAME = "wikimedia/wikipedia"
MAX_ARTICLES = 2000 # 필요 시 증설

# 청킹 설정
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

# 임베딩/LLM
EMBED_MODEL = "intfloat/multilingual-e5-base"
LLM_MODEL   = "Qwen/Qwen2.5-7B-Instruct"

# 인덱싱 배치
BATCH_ADD_ARTICLES = 600 # 문서 600개마다 flush
ADD_DOCS_MINI_BATCH = 64 # Chroma.add_documents 배치
TOP_K = 5

`Hugging Face`의 `wikimedia/wikipedia`는 다양한 언어를 기준으로 수집되었기 때문에 한국어 데이터셋을 조회해보겠습니다.

In [ ]:
from datasets import load_dataset, get_dataset_config_names

def pick_latest_wikipedia_config(lang: str) -> str:
    configs = get_dataset_config_names(DATASET_NAME)
    lang_cfgs = [c for c in configs if c.endswith(f".{lang}")]
    if not lang_cfgs:
        raise ValueError(f"No configs for lang='{lang}' in {DATASET_NAME}")
    def date_key(c):
        m = re.match(r"^(\d{8})\.", c)   # 'YYYYMMDD.lang'
        return int(m.group(1)) if m else -1
    return sorted(lang_cfgs, key=date_key)[-1]

LATEST_CFG = pick_latest_wikipedia_config(LANG)
print("Using dataset config:", LATEST_CFG)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Using dataset config: 20231101.ko


`wikimedia/wikipedia` 데이터셋은 각 언어별로 60~70만개 가량의 문서로 이루어져있습니다.  
때문에 모든 문서를 조회하고 변환하는 것이 아닌 `streaming`기능을 활용해 위에서 선택한 데이터셋 중 일부만 로드하여 확인해보겠습니다.

In [ ]:
def filtered_take(iterable: Iterable[Dict], limit: int):
    count = 0
    for ex in iterable:
        title = ex.get("title") or ""
        text = (ex.get("text") or "").strip()
        if not text:
            continue
        yield ex
        count += 1
        if count >= limit:
            break

stream_probe = load_dataset(DATASET_NAME, LATEST_CFG, split="train", streaming=True)
probe = list(filtered_take(stream_probe, min(5, MAX_ARTICLES)))
[(e.get("title"), len((e.get("text") or ""))) for e in probe]

[('지미 카터', 3608), ('수학', 5087), ('수학 상수', 872), ('문학', 3340), ('나라 목록', 2790)]

`20231101.ko` 문서에선 다음과 같은 결과가 나옵니다.

```plan text
[('지미 카터', 3608), ('수학', 5087), ('수학 상수', 872), ('문학', 3340), ('나라 목록', 2790)]
```
`지미 카터` 문서의 텍스트 길이가 3608 글자,  
`수학` 문서의 텍스트 길이가 5087 글자,  
`수학 상수` 문서의 텍스트 길이가 872 글자,  
`문학` 문서의 텍스트 길이가 3340 글자,  
`나라 목록` 문서의 텍스트 길이가 2790 글자  
가 조회되고 있습니다.

이번엔 벡터 DB를 생성해보겠습니다. `LangChain`을 사용할 것이기 때문에 벡터 DB 역시 `LangChain`의 `Document`로 변환해주어야합니다.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from urllib.parse import quote
from typing import Dict, List

# 임베딩
emb = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

# 청킹
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

# 일부 데이터에는 url 필드가 비어있어 url을 생성해 대체합니다.
def title_to_url(title: str, lang: str="ko") -> str:
    t = (title or "").replace(" ", "_")
    return f"https://{lang}.wikipedia.org/wiki/{quote(t)}"

def to_documents(example: Dict, lang: str) -> List[Document]:
    title = example.get("title") or "Wikipedia"
    text  = example.get("text") or ""
    url   = example.get("url") or title_to_url(title, lang)

    base_meta = {
        "title": title,
        "url": url,
        "source": url,
        "lang": lang
    }

    chunks = splitter.split_text(text)

    # e5 임베딩 모델은 접두사 규약이 있습니다. 이를 추가합니다.
    return [
        Document(page_content="passage: " + c, metadata=base_meta)
        for c in chunks
    ]

# 벡터DB
def build_faiss(docs: List[Document]) -> FAISS:
    return FAISS.from_documents(docs, emb)

# 배치로 쪼개 청크 문서를 처리합니다.
def add_in_batches_faiss(vdb: FAISS, docs: List[Document], batch_size=256):
    for i in range(0, len(docs), batch_size):
        vdb.add_documents(docs[i:i+batch_size])

/tmp/ipykernel_13615/42346636.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  emb = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


문서를 임베딩 벡터(인덱스)로 변환하겠습니다.  
벡터 DB를 구축하는 작업은 처음 1회만 진행하면 되지만 시간이 오래 걸리기 때문에 여기에서는 3000개의 문서만 작업하겠습니다.  
만일 추가하고 싶다면 `MAX_ARTICLES`를 조정하여야합니다.

In [ ]:
dataset = load_dataset(DATASET_NAME, LATEST_CFG, split="train")

all_docs = []

for ex in filtered_take(dataset, MAX_ARTICLES):
    all_docs.extend(to_documents(ex, LANG))

vectordb = build_faiss(all_docs)

검색기는 다음과 같이 생성합니다.

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": TOP_K})

답변을 생성할 LLM 모델입니다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = 0 if torch.cuda.is_available() else -1
print("CUDA available:", torch.cuda.is_available())

tok = AutoTokenizer.from_pretrained(LLM_MODEL, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

gen_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    max_new_tokens=512,
    do_sample=False,
    pad_token_id=tok.eos_token_id,
    device_map="auto" if torch.cuda.is_available() else None
)

CUDA available: True


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


`Qwen instructed`와 같은 챗봇 용도로 fine-tuning된 모델은 프롬프트를 포멧팅해 생성을 돕는 헬퍼 함수가 필요합니다.  
여기에서는 간단한 instruct를 추가하고 출력에서 프롬프트를 잘라내고 답변만 추출하는 헬퍼 함수를 구현하겠습니다.

In [ ]:
def hf_chat(prompt: str) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer in Korean concisely. Cite sources if provided."},
        {"role": "user", "content": prompt},
    ]
    chat_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = gen_pipe(chat_text)[0]["generated_text"]
    return out[len(chat_text):].strip()

이번엔 LangGraph의 상태 그래프를 정의하고 RAG의 동작 프로세스를 구성하겠습니다.  
먼저 다음과 같이 그래프의 상태 Schema를 정의하겠습니다.

- question : 사용자 질문
- context : 검색으로 가져온 문서 청크들
- citations : 출처(제목/URL)
- answer : 생성된 답변
- grounded : 답변이 컨텍스트에 충분히 근거하는지 여부
- refine_tries : 보정 시도 횟수


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class State(TypedDict):
    question: str
    context: List[str]
    citations: List[str]
    answer: str
    grounded: bool
    refine_tries: int

`LangGraph`는 상태(State)를 입력 받아 여러 노드를 거치면서 변환하는 그래프를 만듭니다.  
먼저 `retrieve` 역할의 노드를 생성하겠습니다.

In [ ]:
def _fmt_cite(meta):
    title = meta.get("title") or "Wikipedia"
    url   = meta.get("url") or ""
    return f"{title} ({url})" if url else title

def retrieve_node(state: State) -> State:
    query = "query: " + state["question"]
    docs = retriever.invoke(query)
    ctx = [d.page_content for d in docs]
    # 출력 시 접두사 제거
    ctx = [c.replace("passage: ", "", 1) if c.startswith("passage: ") else c for c in ctx]
    cites = [_fmt_cite(d.metadata) for d in docs]
    return {"context": ctx, "citations": cites}

이번엔 `generate` 역할을 수행할 노드를 추가하겠습니다.

In [ ]:
GEN_PROMPT = """다음 Context만 사용하여 질문에 답하시오.
- 모를 경우 모른다고 답하시오.
- 간결한 bullet 위주, 마지막에 '출처' 포함.

질문: {question}

Context:
{context}
"""

def generate_node(state: State) -> State:
    context_text = "\n\n---\n\n".join(state["context"][:6]) if state["context"] else "없음"
    # 위에 작성한 GEN_PROMPT를 덧붙입니다.
    prompt = GEN_PROMPT.format(question=state["question"], context=context_text)
    out = hf_chat(prompt) # Qwen 채팅 포멧에 맞춰 호출합니다.
    return {"answer": out}

이번엔 근거를 판정할 `grade` 노드를 작성하겠습니다.

In [ ]:
GRADE_PROMPT = """아래 '답변'이 'Context'에 충분히 근거하는지 단답(YES/NO)으로만 판단하세요.

[답변]
{answer}

[Context]
{context}
"""

def grade_node(state: State) -> State:
    snippet = "\n\n".join(state["context"][:4])
    g = hf_chat(GRADE_PROMPT.format(answer=state["answer"], context=snippet)).upper()
    return {"grounded": ("YES" in g[:3])}

근거 부족시 주어진 context 내 문장을 인용해 답변하도록 `refine` 노드를 추가하겠습니다.

In [ ]:
REFINE_PROMPT = """'답변'을 Context의 내용으로만 재작성하세요.
- Context에 없는 내용은 삭제
- 핵심 문장은 따옴표로 인용
- 간결한 한국어 bullet + '출처' 목록

[답변]
{answer}

[Context]
{context}
"""

def refine_node(state: State) -> State:
    ctx = "\n\n---\n\n".join(state["context"][:8]) if state["context"] else "없음"
    out = hf_chat(REFINE_PROMPT.format(answer=state["answer"], context=ctx))
    # 보정 시도 횟수 +1
    tries = int(state.get("refine_tries", 0)) + 1
    return {"answer": out, "refine_tries": tries}

적절한 Context가 없는 경우를 위해 다음과 같이 `fallback` 노드를 추가하겠습니다.

In [ ]:
FALLBACK_PROMPT = """컨텍스트가 충분하지 않으므로, 일반 지식에 기반해서 답변하세요.
- 답변 맨 앞에 "※ 다음은 일반 지식에 기반한 개략적 답변입니다." 문구를 넣으세요.
- Context에 맞는 부분이 있다면 '추가 참고' 섹션에 분리하세요.
- 간결한 한국어 bullet 위주로 작성하세요.

질문: {question}

[Context]
{context}
"""

def fallback_node(state: State) -> State:
    ctx = "\n\n---\n\n".join(state["context"][:6]) if state["context"] else "없음"
    out = hf_chat(FALLBACK_PROMPT.format(question=state["question"], context=ctx))
    return {"answer": out, "grounded": True}

마지막으로 위에서 선언한 노드를 그래프 구조로 통합하겠습니다.  
이 그래프는 다음과 같이 동작합니다.
- `retrieve` -> `generate` -> `grade`
- `grade`에서 통과하면 `END`
- 통과하지 못할 경우 `refine`에서 보충
- `refine`에서 적절한 답변을 생성 못했을 경우 LLM 자체적으로 답변 생성

In [ ]:
def decide_after_grade(state: State):
    # 근거 충분 -> 종료
    if state.get("grounded"):
        return END
    # 근거 부족 -> refine으로
    return "refine"

In [ ]:
def decide_after_refine(state: State):
    # 이미 refine을 N번 시도했는데도 grounded가 False라면 → fallback 또는 종료
    tries = int(state.get("refine_tries", 0))
    if not state.get("grounded") and tries >= MAX_REFINE_TRIES:
        return "fallback" if ENABLE_FALLBACK else END
    # 보정 답변에 대해 다시 근거 판정하러 grade로 보냄
    return "grade"

In [ ]:
graph = StateGraph(State)
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_node("grade", grade_node)
graph.add_node("refine", refine_node)
graph.add_node("fallback", fallback_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "grade")

graph.add_conditional_edges("grade", decide_after_grade, {"refine": "refine", END: END})
graph.add_conditional_edges("refine", decide_after_refine, {"fallback": "fallback", "grade": "grade", END: END})

app = graph.compile()
print("LangGraph RAG ready")

LangGraph RAG ready


그럼 동작을 테스트해보겠습니다.  
RAG의 동작을 직관적으로 표현하기 위해 처리 속도를 위한 부분을 제외해 추론 시간이 오래걸릴 수 있습니다.  
만일 너무 오래 걸린다면 `ENABLE_FALLBACK`을 False로 두고 벡터 DB에 저장된 문서의 수를 줄여서 다시 시도해보십시오.

In [ ]:
MAX_REFINE_TRIES = 1 # 최대 보정 횟수
ENABLE_FALLBACK = True # refine 실패 시 fallback 노드를 활성화할지 여부

In [ ]:
q1 = "트랜스포머 모델의 핵심 아이디어를 간단히 설명하고 관련 위키 문서 제목도 알려줘."
res = app.invoke({"question": q1, "context": [], "citations": [], "answer": "", "grounded": False, "refine_tries": 0})
print(res["answer"])

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


※ 다음은 일반 지식에 기반한 개략적 답변입니다.

- 트랜스포머 모델의 핵심 아이디어:
  - 멀티헤드 어텐션 메커니즘을 통해 입력 시퀀스의 모든 요소 간의 관계를 효과적으로 학습한다.
  - 순방향 계층 구조를 통해 순서 정보를 유지하면서 병렬 처리가 가능하다.
  - positional encoding을 통해 시퀀스의 순서 정보를 인코딩한다.

추가 참고:
- 트랜스포머 모델의 위키 문서 제목: [Transformer (Neural network)](https://en.wikipedia.org/wiki/Transformer_(neural_network))


In [ ]:
ENABLE_FALLBACK = False

q2 = "무한 급수 수열의 핵심 아이디어를 간단히 설명하고 관련 위키 문서 제목을 함께 알려줘."
res = app.invoke({"question": q2, "context": [], "citations": [], "answer": "", "grounded": False})
print(res["answer"])

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- 무한 급수의 핵심 아이디어는 무한히 많은 항들의 합을 생각하며, 그 합이 유한한 값으로 수렴하는지 여부를 판단하는 것입니다. 수렴하면 그 값이 급수의 합으로 정의됩니다.
- 관련 위키 문서 제목: "무한급수"

출처: Context 제공 내용
